In [ ]:
%cd ..

In [1]:
from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
import sys
import os
sys.path.insert(0, r"C:\Users\namtv40\Projects\prefecthq-external-ingestion\ingestions")

In [2]:
import os
import sys
import json
import time
import requests
import pandas as pd
from datetime import datetime
from requests.auth import HTTPBasicAuth
from common.crawler_util import build_basic_auth_header

In [ ]:
# API_KEY = read_hdfs_https(API_KEY_PATH).strip()
FRESHDESK_API_KEY= os.getenv("FRESHDESK_API_KEY")

if not FRESHDESK_API_KEY:
    raise Exception("Not found FRESHDESK_API_KEY")

# API_COOKIE = read_hdfs_https(API_COOKIE_PATH).strip()

# =========================
# Load config
# =========================
# CONFIG_PATH = "/opt/datasets/crawlers/vcs/freshworks/configs/resources.json"
# cfg = json.loads(read_hdfs_https(CONFIG_PATH))
with open(r".\resources\resources-cx-cso.json", "r") as f:
    cfg = json.loads(f.read().strip())

RESOURCE_NAME = "fact_cso_tickets"
if RESOURCE_NAME not in cfg:
    raise Exception("Resource {} not found in config".format(RESOURCE_NAME))

rc = cfg[RESOURCE_NAME]

HDFS_BASE = rc["hdfs_base"]
STATE_PATH = rc["state_path"]
BASE_URL = rc["base_url"]
RESOURCE_URL = rc["resource_url"]
API_KEY_PATH = rc["api_key_path"]
API_COOKIE_PATH = rc["api_cookie_path"]
QUERY_PARAMS = rc["query_params"]
ENABLE_STATE = rc.get("enable_state", False)

HEADERS = {
    "Authorization": build_basic_auth_header(FRESHDESK_API_KEY, "X"),
}

In [4]:
from common.freshdesk_cso_crawler import *

In [5]:
TABLES = [
     "fact_cso_tickets",
    "dim_cso_companies",
    # "dim_cso_ticket_fields", 
        #  "dim_cso_email_configs",
        #   "dim_cso_company_fields",
          "dim_cso_agents",
          "dim_cso_contacts",
        #   "dim_cso_roles",
        #   "dim_cso_groups",
        #   "dim_cso_admin_group",
        #   "dim_cso_time_entries",
        #   "dim_cso_sla_policies",
        # "dim_cso_contact_fields",
]
LONG_TABLE = []

for resource_name in TABLES:
    out_of_data = False
    for j in range(20):
        for i in range(6):
            NUMBER_OF_PAGE = 50
            try:
                out_of_data = fetch_resource_name_freshdesk(resource_name, cfg, headers=HEADERS, max_page=(i+1) * NUMBER_OF_PAGE, start_page=i*NUMBER_OF_PAGE +1)
                if out_of_data :
                    print("out_of_data at " , i)
                    break
            except Exception as e:
                print(e)
        if out_of_data :
            print("out_of_data at " , j)
            break

Start crawl :  fact_cso_tickets
fact_cso_tickets
Last state = 2026-01-22T19:38:16Z
Request to  https://vcs-care.freshdesk.com/api/v2/tickets {'page': 1, 'per_page': 100, 'include': 'company,stats,requester,description', 'updated_since': '2026-01-22T19:38:16Z'}
Request to  https://vcs-care.freshdesk.com/api/v2/tickets {'page': 2, 'per_page': 100, 'include': 'company,stats,requester,description', 'updated_since': '2026-01-22T19:38:16Z'}
Request to  https://vcs-care.freshdesk.com/api/v2/tickets {'page': 3, 'per_page': 100, 'include': 'company,stats,requester,description', 'updated_since': '2026-01-22T19:38:16Z'}
Request to  https://vcs-care.freshdesk.com/api/v2/tickets {'page': 4, 'per_page': 100, 'include': 'company,stats,requester,description', 'updated_since': '2026-01-22T19:38:16Z'}
Request to  https://vcs-care.freshdesk.com/api/v2/tickets {'page': 5, 'per_page': 100, 'include': 'company,stats,requester,description', 'updated_since': '2026-01-22T19:38:16Z'}
Request to  https://vcs-car